# MRDRP Dashboard Launcher — Backend MR for any analysis set (P2)

Current stage only: deploys the two pending pieces for the new "Run MR for a saved analysis set" feature (`mr_pipeline.py` + the Backend MR Results page patch), then launches the dashboard. It assumes `app.py`, `gwas_catalog_client.py`, and `gwas_catalog_ftp.py` are already in place from earlier stages -- this notebook does not touch the GWAS Catalog Search page or re-apply any of that history.

**Important -- two separate notebooks, two separate jobs:**
- **This notebook** only needs plain Python + Streamlit. It does **not** need R, rpy2, TwoSampleMR, or an OpenGWAS token -- the dashboard only *reads* result files, it never runs the MR computation itself.
- **`Python_based_MR_drug_repurposing_pipeline.ipynb`** (your separate, existing notebook) is where the actual R/TwoSampleMR computation runs, using `mr_pipeline.run_pipeline_for_analysis_set(...)` -- that notebook still needs all of its existing R/rpy2/OpenGWAS setup cells.

Cells below are kept small/granular on purpose so a slow step doesn't stall the whole run.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
from pathlib import Path

project_path = Path("/content/drive/MyDrive/UM_WQF7023/MRDRP-main")
app_path = project_path / "app.py"

print("Project path:", project_path)
print("app.py exists:", app_path.exists())

Project path: /content/drive/MyDrive/UM_WQF7023/MRDRP-main
app.py exists: True


## 2. Sanity-check earlier stages are already in place

This notebook builds on top of the GWAS Catalog Search stage. If any of these are missing, go back and run that stage's deployment cells first -- this notebook does not recreate them.

In [ ]:
required_files = ["app.py", "gwas_catalog_client.py", "gwas_catalog_ftp.py", "analysis_set_record.csv"]
missing = [f for f in required_files if not (project_path / f).exists()]

if missing:
    raise FileNotFoundError(
        f"Missing from {project_path}: {', '.join(missing)}. "
        "Please complete the earlier GWAS Catalog Search / Analysis Set Selection stages first."
    )

print("All required files from earlier stages are present.")

All required files from earlier stages are present.


## 5. Install the dashboard's Python dependencies

Just Streamlit + pyliftover (used by the FTP-download page's automatic liftover step) -- no R needed here. Both need reinstalling every fresh Colab runtime, since nothing but Drive persists across sessions.

In [ ]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 100.8 MB/s eta 0:00:00


In [ ]:
!pip install -q pyliftover --break-system-packages

## 6. Launch the dashboard

In [ ]:
!pkill -f streamlit || true
!pkill -f cloudflared || true

^C
^C


In [ ]:
!streamlit run /content/drive/MyDrive/UM_WQF7023/MRDRP-main/app.py --server.port 8501 > /content/streamlit.log 2>&1 &

In [ ]:
import time
time.sleep(5)

!cat /content/streamlit.log



2026-08-22 04:04:26.355 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.19.52.131:8501



In [ ]:
!curl -I http://localhost:8501

HTTP/1.1 200 OK
date: Sat, 22 Aug 2026 04:04:36 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 11141
last-modified: Sat, 22 Aug 2026 04:03:57 GMT
etag: "2bf0d62fe17cb3e60d0ebbc0609e3973"
cache-control: no-cache



## 7. Expose the dashboard via a Cloudflare Tunnel

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

In [ ]:
!nohup /content/cloudflared tunnel --url http://localhost:8501 > /content/streamlit_tunnel.log 2>&1 &

In [ ]:
import time
import re

time.sleep(8)

with open("/content/streamlit_tunnel.log", "r", encoding="utf-8", errors="ignore") as f:
    log_text = f.read()

print(log_text)

urls = re.findall(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", log_text)

if len(urls) == 0:
    raise RuntimeError("No trycloudflare URL found. Please wait a few seconds and run this cell again.")

streamlit_url = urls[0]
print("Streamlit public URL:")
print(streamlit_url)

2026-08-22T04:04:47Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-22T04:04:47Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-22T04:04:51Z INF +--------------------------------------------------------------------------------------------+
2026-08-22T04:04:51Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-22T04:04:51Z INF |  https://glucose-spice-enabled-forth.trycloudflare.com

## 8. Test this stage

Open the URL printed above, click **"Backend MR Results"** in the sidebar, and look for the new **"0) Run MR for a saved analysis set (P2)"** section at the top. Pick one of your saved analysis sets from the dropdown -- it will say results don't exist yet (unless you've already run the pipeline notebook for it) and show you the exact `mr_pipeline.run_pipeline_for_analysis_set(...)` call to paste into `Python_based_MR_drug_repurposing_pipeline.ipynb`.

Once you've run that in the pipeline notebook, come back and refresh this page -- the results, clumping summary, and run summary should appear.

If anything above raised an error, or the new section doesn't show up, paste the traceback / a screenshot back.